In [ ]:
# Install dependencies (run once per fresh environment, before any imports;
# restart the kernel afterwards if packages were upgraded)
%pip install -q -r ../requirements.txt

In [ ]:
# --- Setup (Step 0) ---
import sys
from pathlib import Path

# Make src/ importable whether run locally (cwd=src/) or from repo root
for p in (Path.cwd(), Path.cwd() / "src"):
    if (p / "config.py").is_file() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

import config as C

C.set_seed()
# HF token (gated Gemma 3) is checked with C.get_hf_token() right before loading models
print("repo root:", C.REPO_ROOT)
print("models:", list(C.MODELS))
print("runtime:", C.runtime_info())

# Step 1 – Data: Spider dev, prompts, difficulty

The cells below build `src/spider_dev_inputs.json`: the inputs for every later step, one entry per Spider dev example (all 1034). Each entry has the example's `id` (its position in Spider `dev.json`), `db_id`, `question` and gold `query`, plus two derived fields:

- `prompt`: the zero-shot prompt from `src/prompt_template.txt`, filled with the example's `db_id`, `question`, and its database schema (CREATE TABLE + 3 sample rows). The design rationale is in `docs/prompt_design.md`.
- `difficulty`: easy / medium / hard / extra, computed with the official Spider evaluation script.

Model outputs are **not** stored in this file. Each model's outputs live in its own `outputs/<model>/<run>/responses.jsonl` and are matched to these inputs by `id` (Step 2). No GPU is needed for this step.

## Step 1.1 – Download Spider dev + databases

This step downloads the official Spider 1.0 zip from Google Drive (the link on yale-lily.github.io/spider) with `gdown` and unpacks it to `data/spider/`. We need the zip, not just the GitHub JSON files, because it also contains the SQLite databases (`database/<db_id>/<db_id>.sqlite`). The databases are used for the sample rows in the prompt and later for Execution Accuracy. The download is skipped if the files are already there. If Google Drive refuses the download (quota), download the zip by hand and unzip it so that `data/spider/` holds `dev.json`, `tables.json` and `database/`.

In [ ]:
import json
import zipfile

import gdown


def spider_ready() -> bool:
    return all((C.SPIDER_DIR / f).exists() for f in ("dev.json", "tables.json", "database"))


if not spider_ready():
    if C.SPIDER_DIR.exists():
        raise RuntimeError(f"{C.SPIDER_DIR} exists but is incomplete; remove it and rerun this cell.")
    C.DATA_DIR.mkdir(parents=True, exist_ok=True)
    zip_path = C.DATA_DIR / "spider_data.zip"
    if not zip_path.exists():
        gdown.download(id=C.SPIDER_GDRIVE_ID, output=str(zip_path), quiet=False)
    with zipfile.ZipFile(zip_path) as zf:  # skip the __MACOSX/ junk entries in the zip
        zf.extractall(C.DATA_DIR, members=[m for m in zf.namelist() if m.startswith("spider_data/")])
    (C.DATA_DIR / "spider_data").rename(C.SPIDER_DIR)
    zip_path.unlink()

if not spider_ready():
    raise FileNotFoundError(
        "Spider download failed. Download the zip from https://yale-lily.github.io/spider by hand "
        f"and unzip it so that {C.SPIDER_DIR} contains dev.json, tables.json and database/."
    )

dev = json.loads((C.SPIDER_DIR / "dev.json").read_text())
tables = {t["db_id"]: t for t in json.loads((C.SPIDER_DIR / "tables.json").read_text())}


def db_path(db_id: str) -> Path:
    return C.SPIDER_DIR / "database" / db_id / f"{db_id}.sqlite"


dev_dbs = sorted({ex["db_id"] for ex in dev})
missing = [d for d in dev_dbs if d not in tables or not db_path(d).exists()]
assert len(dev) == 1034, len(dev)
assert not missing, f"missing schema or .sqlite for: {missing}"
print(f"{len(dev)} dev examples over {len(dev_dbs)} databases")

## Step 1.2 – Download the official evaluation script

This step clones `taoyds/test-suite-sql-eval` (the official evaluation script for Spider) and checks out a pinned commit, so difficulty labels and metrics stay reproducible. We import its SQL parser (`process_sql`) and `Evaluator` here so that Step 1.5 can call the script's own difficulty function. The parser tokenizes with nltk, which needs the `punkt_tab` data.

In [ ]:
import subprocess

import nltk

if not (C.EVAL_SCRIPT_DIR / "evaluation.py").exists():
    subprocess.run(["git", "clone", "--quiet", C.EVAL_SCRIPT_REPO, str(C.EVAL_SCRIPT_DIR)], check=True)
subprocess.run(["git", "-C", str(C.EVAL_SCRIPT_DIR), "checkout", "--quiet", C.EVAL_SCRIPT_COMMIT], check=True)
nltk.download("punkt_tab", quiet=True)

if str(C.EVAL_SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(C.EVAL_SCRIPT_DIR))
from evaluation import Evaluator
from process_sql import Schema, get_schema, get_sql

print("eval script at", C.EVAL_SCRIPT_COMMIT[:7], "->", C.EVAL_SCRIPT_DIR)

## Step 1.3 – Schema representation (CREATE TABLE + 3 sample rows)

This step builds the `{schema}` text for each database. For every table it writes:
- a `CREATE TABLE` statement built from `tables.json`, with original table/column names, column types (`others` → `TEXT`), `PRIMARY KEY` and `FOREIGN KEY ... REFERENCES ...` lines;
- a comment block with the first 3 rows (`SELECT * FROM <table> LIMIT 3`), read from the SQLite file. This is the "Create Table + Select 3" format of Rajkumar et al. (2022).

Names that are not plain identifiers (e.g. `Official_ratings_(millions)`) are double-quoted, which is valid SQLite. Cell values are tab-separated: `NULL` stands for missing values, line breaks become spaces, and values longer than `MAX_CELL_CHARS` are truncated. If a table's columns in `tables.json` don't match the database, the cell falls back to `SELECT *` and counts it (expected: 0). There are only 20 dev databases, so each one is rendered once and cached in `schemas`.

In [ ]:
import re
import sqlite3

TYPE_MAP = {"others": "text"}  # tables.json type -> type shown in the DDL
fallbacks: list[str] = []  # tables where the explicit-column SELECT failed


def ident(name: str) -> str:
    """Name as written in the DDL: quoted only if it is not a plain identifier."""
    return name if re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", name) else '"' + name.replace('"', '""') + '"'


def fmt_cell(value) -> str:
    s = "NULL" if value is None else re.sub(r"[\t\r\n]+", " ", str(value))
    return s if len(s) <= C.MAX_CELL_CHARS else s[: C.MAX_CELL_CHARS] + "..."


def sample_rows(conn, table: str, columns: list[str]) -> tuple[list[str], list[tuple]]:
    quoted = ", ".join('"' + c.replace('"', '""') + '"' for c in columns)
    try:
        cur = conn.execute(f'SELECT {quoted} FROM "{table}" LIMIT {C.N_SAMPLE_ROWS}')
        return columns, cur.fetchall()
    except sqlite3.OperationalError:
        fallbacks.append(table)
        cur = conn.execute(f'SELECT * FROM "{table}" LIMIT {C.N_SAMPLE_ROWS}')
        return [d[0] for d in cur.description], cur.fetchall()


def render_schema(t: dict, db_file: Path) -> str:
    names, cols, types = t["table_names_original"], t["column_names_original"], t["column_types"]
    pks = {i for pk in t["primary_keys"] for i in (pk if isinstance(pk, list) else [pk])}
    conn = sqlite3.connect(f"file:{db_file}?mode=ro", uri=True)
    conn.text_factory = lambda b: b.decode("utf-8", errors="replace")  # a few DBs have bad UTF-8
    blocks = []
    for ti, table in enumerate(names):
        idx = [i for i, (tj, _) in enumerate(cols) if tj == ti]
        lines = [
            f"  {ident(cols[i][1])} {TYPE_MAP.get(types[i], types[i]).upper()}" + (" PRIMARY KEY" if i in pks else "")
            for i in idx
        ]
        lines += [
            f"  FOREIGN KEY ({ident(cols[a][1])}) REFERENCES {ident(names[cols[b][0]])}({ident(cols[b][1])})"
            for a, b in t["foreign_keys"]
            if cols[a][0] == ti
        ]
        header, rows = sample_rows(conn, table, [cols[i][1] for i in idx])
        sample = [
            "/*",
            f"{C.N_SAMPLE_ROWS} example rows:",
            f"SELECT * FROM {ident(table)} LIMIT {C.N_SAMPLE_ROWS};",
            "\t".join(header),
            *("\t".join(fmt_cell(v) for v in row) for row in rows),
            "*/",
        ]
        blocks.append(f"CREATE TABLE {ident(table)} (\n" + ",\n".join(lines) + "\n);\n" + "\n".join(sample))
    conn.close()
    return "\n\n".join(blocks)


schemas = {db_id: render_schema(tables[db_id], db_path(db_id)) for db_id in dev_dbs}
for db_id, text in schemas.items():  # one CREATE TABLE per table in tables.json
    assert text.count("CREATE TABLE ") == len(tables[db_id]["table_names_original"]), db_id
print(f"rendered {len(schemas)} schemas; SELECT * fallbacks: {len(fallbacks)} {fallbacks}")
print(schemas["concert_singer"])

## Step 1.4 – Add the `prompt` key

This step fills the plain-text template (`src/prompt_template.txt`) for every dev example and stores the result under a new `prompt` key, next to `question` and `db_id`. The template has exactly three placeholders: `{db_id}`, `{schema}` (from Step 1.3) and `{question}`. All models will get this same prompt as a single user message. We check that no placeholder is left unfilled and that every prompt contains its own question.

In [ ]:
import statistics

template = C.PROMPT_TEMPLATE.read_text()

for ex in dev:
    ex["prompt"] = template.format(db_id=ex["db_id"], schema=schemas[ex["db_id"]], question=ex["question"])

for ex in dev:
    assert not any(p in ex["prompt"] for p in ("{db_id}", "{schema}", "{question}")), ex["question"]
    assert ex["question"] in ex["prompt"]

lengths = [len(ex["prompt"]) for ex in dev]
print(f"prompt length (chars): min {min(lengths)}, median {statistics.median(lengths):.0f}, max {max(lengths)}")
print("=" * 80)
print(dev[0]["prompt"])

## Step 1.5 – Add the `difficulty` key (official hardness)

This step labels each example easy / medium / hard / extra using the official script's `Evaluator.eval_hardness`. We follow `evaluation.py` exactly: parse the gold SQL with the database schema (`get_sql(Schema(get_schema(db)), query)`), then classify the parsed query by counting its SQL components (joins, GROUP BY, nesting, ...). The resulting counts must match the published Spider dev split (248 / 446 / 174 / 166). That confirms the labels were reproduced correctly.

In [ ]:
from collections import Counter

evaluator = Evaluator()
eval_schemas = {db_id: Schema(get_schema(str(db_path(db_id)))) for db_id in dev_dbs}

for ex in dev:
    ex["difficulty"] = evaluator.eval_hardness(get_sql(eval_schemas[ex["db_id"]], ex["query"]))

counts = Counter(ex["difficulty"] for ex in dev)
print({level: counts[level] for level in ("easy", "medium", "hard", "extra")})
assert counts == {"easy": 248, "medium": 446, "hard": 174, "extra": 166}, counts

## Step 1.6 – Save the model inputs

This step writes `src/spider_dev_inputs.json`, keeping only what a model or the evaluation needs, one entry per example: `id`, `db_id`, `question`, gold `query`, `difficulty`, `prompt`. Spider's tokenized fields and parsed `sql` are left out; they stay in `data/spider/dev.json`, and the evaluation script reparses the gold SQL anyway. The file is written deterministically, so rerunning Step 1 gives an identical file (same sha256).

In [ ]:
import hashlib

INPUT_FIELDS = ["db_id", "question", "query", "difficulty", "prompt"]
dev_inputs = [{"id": i, **{f: ex[f] for f in INPUT_FIELDS}} for i, ex in enumerate(dev)]

C.DEV_INPUTS.parent.mkdir(parents=True, exist_ok=True)
C.DEV_INPUTS.write_text(json.dumps(dev_inputs, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

digest = hashlib.sha256(C.DEV_INPUTS.read_bytes()).hexdigest()
print(f"wrote {len(dev_inputs)} examples -> {C.DEV_INPUTS} ({C.DEV_INPUTS.stat().st_size / 1e6:.1f} MB)")
print("fields:", list(dev_inputs[0]))
print("sha256:", digest)

# Step 2 – Inference: prompt each model on all dev examples

The cells below prompt all three models zero-shot on every example in `src/spider_dev_inputs.json`. Each model's outputs go to its own `outputs/<model>/<run>/responses.jsonl`, one record per example, keyed by the same `id` as the inputs. The only exception is the pilot (Step 2.5): it is a sanity check, so it is written to the local, gitignored `archive/pilot/`. To keep the comparison fair, every model gets the **same prompt text, decoding settings, output cap, precision and batch size**. The only intended difference is that Gemma 4 E4B reasons before answering.

Every run is **resumable**. Outputs are appended to `outputs/<model>/<run>/responses.jsonl` after each batch, and a rerun skips examples that are already done. The full runs therefore survive interruptions, and rerunning a finished cell costs nothing.

Prerequisite: the Gemma 3 models are gated. Accept their licenses on Hugging Face and provide a token (`huggingface-cli login` or the `HF_TOKEN` env var) before running them.

## Step 2.1 – Inference settings

This cell shows the settings shared by all three models. They come from `src/config.py`, and each run also writes them to its `config.json`.
- **Greedy decoding** (`do_sample=False`) for every model. It's deterministic and reproducible, and it's the usual choice for Text-to-SQL, so temperature, top-p and top-k don't apply. They are passed explicitly as `None`, because E4B's own `generation_config` defaults to sampling.
- **Max output length 8192 tokens** for every model. The Gemma 3 models stop at end-of-turn long before this; the long cap is there for E4B's reasoning.
- **bf16** weights, the same batch size, left padding, one NVIDIA A40. Attention uses PyTorch SDPA, except for **Gemma 3 1B, which uses eager attention**. In the first full run, SDPA's memory-efficient kernel corrupted 6 of 1B's batched outputs (a garbage token instead of SQL); the same examples are correct with eager attention. Details are in `docs/inference_issues.md`.
- **Reasoning:** E4B uses thinking mode (`enable_thinking=True`, which puts `<|think|>` in the system turn). In a probe, E4B **skipped thinking on most easy questions** and answered with bare SQL. Task 2 needs a trace for every example, so we also pre-fill E4B's turn with the thought-channel opener `<|channel>thought\n`, which forces a trace each time. This mirrors DeepSeek-R1's recommendation to force the `<think>\n` opener. The user message is still identical for all models.

In [ ]:
import pandas as pd

settings = pd.DataFrame(
    [
        {
            "model": spec.name,
            "category": spec.category,
            "params": spec.params,
            "loader": spec.loader,
            "precision": C.QUANTIZATION,
            "attention": spec.attn_implementation,
            "decoding": "sampling" if C.DECODING[key].do_sample else "greedy",
            "temperature": C.DECODING[key].temperature or "n/a",
            "top_p": C.DECODING[key].top_p or "n/a",
            "max_new_tokens": C.DECODING[key].max_new_tokens,
            "batch_size": C.BATCH_SIZE,
            "reasoning": "on (forced thought channel)" if spec.reasoning else "none",
        }
        for key, spec in C.MODELS.items()
    ]
).set_index("model")
display(settings)
print("runtime:", C.runtime_info())

## Step 2.2 – Load a model and build its input

`load_model` loads each model with the class its model card uses: `AutoModelForCausalLM` for the text-only Gemma 3 1B, and the multimodal classes for Gemma 3 4B and Gemma 4 E4B, of which only the text path is used. Gated models get the HF token here. `build_input` wraps the example's `prompt` as a single user message in the model's own chat template. For E4B it turns thinking on and adds the thought-channel pre-fill.

In [ ]:
import gc
import time

import torch
import transformers


def load_model(spec: C.ModelSpec):
    token = C.get_hf_token() if spec.gated else None
    proc_cls = transformers.AutoTokenizer if spec.loader == "AutoModelForCausalLM" else transformers.AutoProcessor
    processor = proc_cls.from_pretrained(spec.hf_id, token=token)
    tokenizer = getattr(processor, "tokenizer", processor)
    tokenizer.padding_side = "left"  # decoder-only batching
    model = getattr(transformers, spec.loader).from_pretrained(
        spec.hf_id, dtype=torch.bfloat16, device_map="cuda", attn_implementation=spec.attn_implementation, token=token
    ).eval()
    return model, processor, tokenizer


def build_input(spec: C.ModelSpec, processor, prompt: str) -> str:
    extra = {"enable_thinking": True} if spec.reasoning else {}
    text = processor.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True, **extra
    )
    return text + (C.THINKING_PREFILL if spec.reasoning else "")

## Step 2.3 – Output parsing and SQL extraction

This cell splits each raw output into a reasoning trace and a final answer, then pulls one SQL query out of the answer.
- **E4B:** the output is `<|channel>thought\n<trace><channel|><answer><turn|>`, and the processor's `parse_response` splits it into `thinking` and the answer. If the 8192-token cap cuts the trace off before `<channel|>`, everything generated is kept as the (truncated) trace and the answer is empty.
- **Gemma 3:** the output (without special tokens) is the answer, and there is no trace.

`extract_sql` applies the same rules to every model:
1. If the answer has a Markdown code fence, use the first fenced block.
2. Start at the first `SELECT` (or a `WITH <name> AS (` CTE).
3. Stop at the first `;` or blank line outside a quoted string. This drops any explanation that follows.
4. Collapse all whitespace onto one line, the way the Spider evaluation script expects.

If no query is found, the result is `""` and the output is marked **invalid**. Invalid and truncated outputs are counted as incorrect in the metrics.

In [ ]:
FENCE = re.compile(r"```[ \t]*(?:sql|sqlite)?[ \t]*\n?(.*?)(?:```|$)", re.S | re.I)
SQL_START = re.compile(r"\bSELECT\b|\bWITH\s+\w+\s+AS\s*\(", re.I)


def extract_sql(answer: str) -> str:
    text = answer or ""
    fenced = FENCE.search(text)
    if fenced:
        text = fenced.group(1)
    start = SQL_START.search(text)
    if not start:
        return ""
    text, quote = text[start.start() :], None
    for i, ch in enumerate(text):
        if quote:
            quote = None if ch == quote else quote
        elif ch in "'\"`":
            quote = ch
        elif ch == ";" or text.startswith("\n\n", i):
            text = text[:i]
            break
    return " ".join(text.split())


def split_output(spec: C.ModelSpec, processor, gen_ids: list[int]) -> tuple[str, str | None, str]:
    """Return (raw_output, thinking, answer) for one example's generated token ids."""
    if not spec.reasoning:
        raw = processor.decode(gen_ids, skip_special_tokens=False)
        return raw, None, processor.decode(gen_ids, skip_special_tokens=True).strip()
    raw = C.THINKING_PREFILL + processor.decode(gen_ids, skip_special_tokens=False)
    if "<channel|>" not in raw:  # cap reached while still thinking
        trace = processor.decode(gen_ids, skip_special_tokens=True)
        return raw, trace.strip(), ""
    parsed = processor.parse_response(raw, prefix="")
    return raw, (parsed.get("thinking") or "").strip(), (parsed.get("content") or "").strip()


demo = [
    "SELECT count(*) FROM singer",
    "```sql\nSELECT name\nFROM singer\nWHERE age > 30;\n```\nThis query lists singers older than 30.",
    "Here is the query: SELECT name FROM singer WHERE country = 'a;b';  -- done",
    "SELECT T1.name FROM singer AS T1\n\nThis joins nothing.",
    "I cannot answer this question.",
]
for d in demo:
    print(f"{d!r:95} -> {extract_sql(d)!r}")

## Step 2.4 – Resumable generation loop

`run_model(key, ids, run_name)` generates outputs for the given dev example ids. It runs in batches of `C.BATCH_SIZE`, with examples sorted by prompt length to reduce padding. After each batch it appends one JSON line per example to the run's `responses.jsonl` (`outputs/<model>/<run_name>/` for full runs, `archive/pilot/<model>/` for the pilot; see `C.run_path`), and on a rerun it skips ids that are already in that file. Batches are fixed from the full sorted id list, and an incomplete batch is regenerated whole while only its missing ids are written. A resumed run therefore produces exactly the same outputs as an uninterrupted one. This matters because bf16 results on padded batches depend slightly on which examples share a batch. Each record holds:
- `query` (extracted SQL), `answer`, `raw_output` (full decoded text, special tokens included) and `thinking` (E4B trace, or `null`)
- `valid` and `truncated` (no end-of-turn token before the cap)
- `input_tokens`, `output_tokens` (generated tokens up to end-of-turn, padding excluded) and `thinking_tokens`
- `latency_s`: the batch's wall-clock time divided by the batch size, i.e. the amortized seconds per example
- `peak_mem_gb`

The run's settings and runtime are written to `config.json` next to the outputs.

In [ ]:
from tqdm.auto import tqdm


def read_records(key: str, run_name: str) -> dict[int, dict]:
    path = C.run_path(key, run_name) / "responses.jsonl"
    if not path.exists():
        return {}
    return {r["id"]: r for r in map(json.loads, path.read_text().splitlines()) if r}


def run_model(key: str, ids: list[int], run_name: str) -> None:
    spec, dec = C.MODELS[key], C.DECODING[key]
    out_path = C.run_dir(key, run_name) / "responses.jsonl"
    done = read_records(key, run_name)
    # Batches are fixed from the full id list, so batch composition (and hence the bf16
    # numerics) is the same whether or not the run was interrupted and resumed.
    ordered = sorted(ids, key=lambda i: (len(dev[i]["prompt"]), i))
    batches = [ordered[b : b + C.BATCH_SIZE] for b in range(0, len(ordered), C.BATCH_SIZE)]
    batches = [batch for batch in batches if any(i not in done for i in batch)]
    n_todo = sum(i not in done for i in ids)
    print(f"{spec.name} [{run_name}]: {len(ids) - n_todo}/{len(ids)} done, {n_todo} to go")
    if not batches:
        return
    C.save_run_config(
        key, run_name, batch_size=C.BATCH_SIZE, n_examples=len(ids),
        prompt_template=str(C.PROMPT_TEMPLATE.relative_to(C.REPO_ROOT)),
        thinking_prefill=C.THINKING_PREFILL if spec.reasoning else None,
    )
    C.set_seed()
    model, processor, tokenizer = load_model(spec)
    eos_ids = model.generation_config.eos_token_id
    eos_ids = set(eos_ids if isinstance(eos_ids, list) else [eos_ids])
    torch.cuda.reset_peak_memory_stats()

    with out_path.open("a", encoding="utf-8") as f:
        for batch in tqdm(batches, desc=spec.name):
            texts = [build_input(spec, processor, dev[i]["prompt"]) for i in batch]
            enc = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            with torch.inference_mode():
                out = model.generate(
                    **enc, max_new_tokens=dec.max_new_tokens, do_sample=dec.do_sample,
                    temperature=dec.temperature, top_p=dec.top_p, top_k=dec.top_k,
                )
            torch.cuda.synchronize()
            wall = time.perf_counter() - t0
            peak = torch.cuda.max_memory_allocated() / 1e9
            width = enc["input_ids"].shape[1]
            for j, i in enumerate(batch):
                if i in done:  # regenerated only to keep the batch identical
                    continue
                gen = [t for t in out[j, width:].tolist() if t != tokenizer.pad_token_id]
                raw, thinking, answer = split_output(spec, processor, gen)
                query = extract_sql(answer)
                rec = {
                    "id": i,
                    "query": query,
                    "answer": answer,
                    "raw_output": raw,
                    "thinking": thinking,
                    "valid": bool(query),
                    "truncated": not gen or gen[-1] not in eos_ids,
                    "input_tokens": int(enc["attention_mask"][j].sum()),
                    "output_tokens": len(gen),
                    "thinking_tokens": len(tokenizer(thinking, add_special_tokens=False)["input_ids"]) if thinking else 0,
                    "latency_s": wall / len(batch),
                    "batch_wall_s": wall,
                    "batch_size": len(batch),
                    "peak_mem_gb": round(peak, 2),
                }
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            f.flush()

    del model, processor, tokenizer
    gc.collect()
    torch.cuda.empty_cache()


def summarize(key: str, run_name: str, n_full: int = len(dev)) -> dict:
    recs = list(read_records(key, run_name).values())
    total = sum(r["latency_s"] for r in recs)
    return {
        "model": C.MODELS[key].name,
        "n": len(recs),
        "s/example": round(total / len(recs), 2),
        "mean output tokens": round(statistics.mean(r["output_tokens"] for r in recs), 1),
        "max output tokens": max(r["output_tokens"] for r in recs),
        "mean thinking tokens": round(statistics.mean(r["thinking_tokens"] for r in recs), 1),
        "truncated": sum(r["truncated"] for r in recs),
        "invalid": sum(not r["valid"] for r in recs),
        "peak GPU mem (GB)": max(r["peak_mem_gb"] for r in recs),
        "projected full run (h)": round(total / len(recs) * n_full / 3600, 2),
    }

## Step 2.5 – Pilot run (30 examples per model)

Before the full runs, each model runs on a small pilot set of **30 dev examples**. The examples are sampled with seed 42 and stratified by difficulty in proportion to the full dev set, and their ids are saved to `archive/pilot/pilot_ids.json`. The pilot outputs go to `archive/pilot/<model>/` (`responses.jsonl` + `config.json`). That folder is local only and not committed: the pilot is a sanity check, not a reported result. Rerunning this cell recreates it, with the same ids and the same outputs. The pilot measures seconds per example, output length, truncation and invalid rates, and peak GPU memory, then projects the full-run time. These numbers tell us whether the batch size and the 8192-token cap need to change before the full runs.

In [ ]:
import random
from collections import defaultdict

N_PILOT = 30
by_level = defaultdict(list)
for i, ex in enumerate(dev):
    by_level[ex["difficulty"]].append(i)
# proportional allocation (largest remainder), then a seeded sample within each level
quota = {lvl: N_PILOT * len(v) / len(dev) for lvl, v in by_level.items()}
alloc = {lvl: int(q) for lvl, q in quota.items()}
for lvl in sorted(quota, key=lambda l: quota[l] - alloc[l], reverse=True)[: N_PILOT - sum(alloc.values())]:
    alloc[lvl] += 1
rng = random.Random(C.SEED)
pilot_ids = sorted(i for lvl in ("easy", "medium", "hard", "extra") for i in rng.sample(by_level[lvl], alloc[lvl]))
C.PILOT_IDS.parent.mkdir(parents=True, exist_ok=True)
C.PILOT_IDS.write_text(json.dumps({"seed": C.SEED, "alloc": alloc, "ids": pilot_ids}, indent=2))
print("pilot allocation:", alloc)

for key in C.MODELS:
    run_model(key, pilot_ids, "pilot")

display(pd.DataFrame([summarize(key, "pilot") for key in C.MODELS]).set_index("model"))
for key, spec in C.MODELS.items():
    recs = read_records(key, "pilot")
    for i in pilot_ids[:2]:
        r = recs[i]
        print("=" * 100, f"\n{spec.name} | id {i} ({dev[i]['difficulty']}) | gold: {dev[i]['query']}")
        print("pred:", r["query"])
        print("raw output:", r["raw_output"][:1500])

## Step 2.6 – Full runs (all 1034 dev examples)

This cell runs each model on the full dev set (run name `baseline`), starting with E4B, the longest run. Because the runs are resumable, re-running this cell after an interruption continues where it stopped. Once a model is complete, its run is skipped.

In [ ]:
ALL_IDS = list(range(len(dev)))
for key in ("gemma4-e4b", "gemma3-4b", "gemma3-1b"):
    run_model(key, ALL_IDS, "baseline")

display(pd.DataFrame([summarize(key, "baseline") for key in C.MODELS]).set_index("model"))

# Step 3 – Evaluation: how good are the generated queries?

The cells below score every model's SQL against the gold (correct) SQL for all 1034 dev examples, overall and by difficulty. They also compare the models with each other, count invalid outputs, and summarize speed and output length. All tables are written to `outputs/metrics/` (CSV) and the plots to `outputs/metrics/figures/`. **The central results table is `outputs/metrics/task1_summary.csv`** (Step 3.4).

**Metrics, in short** (full explanation in `docs/evaluation_metrics.md`):

| Metric | Question it answers | Source |
|---|---|---|
| **Exact Match (EM)** | Is the query *built* the same way as the gold query? | Spider paper (Yu et al., 2018) |
| **Component F1** | How well does each *part* of the query (SELECT, WHERE, …) match? | Spider paper |
| **Execution Accuracy (EX)** | Does the query return the *same answer* when run on the database? | official Spider evaluation script / leaderboard |
| **EM (normalized)** | EM again, after rewriting `INNER JOIN` → `JOIN` | our addition (see Step 3.2) |

All scores come from the **official Spider evaluation code** (`test-suite-sql-eval`, pinned commit), and Step 3.3 checks that our numbers equal what the official script prints.

## Step 3.1 – Load inputs and outputs, and check they line up

This cell loads the inputs (`src/spider_dev_inputs.json`) and each model's full-run outputs (`outputs/<model>/baseline/responses.jsonl`) and matches them by `id`. Before scoring, it checks that:
- every model has exactly one output for every input example (no missing, extra or duplicate ids);
- Gemma 4 E4B has a reasoning trace for every example (Task 2 analyzes these).

In [ ]:
inputs = {ex["id"]: ex for ex in json.loads(C.DEV_INPUTS.read_text())}
runs = {key: read_records(key, "baseline") for key in C.MODELS}

for key, recs in runs.items():
    assert set(recs) == set(inputs), f"{C.MODELS[key].name}: outputs don't match the inputs; rerun Step 2.6"
(REASONING_KEY,) = [key for key, spec in C.MODELS.items() if spec.reasoning]
assert all(r["thinking"] for r in runs[REASONING_KEY].values()), "some E4B examples have no reasoning trace"

MODEL_NAMES = [spec.name for spec in C.MODELS.values()]  # fixed order for every table and plot
LEVELS = ["easy", "medium", "hard", "extra", "all"]
print(f"{len(inputs)} examples; outputs complete for: {', '.join(MODEL_NAMES)}; "
      f"{C.MODELS[REASONING_KEY].name} has {len(runs[REASONING_KEY])} reasoning traces")

## Step 3.2 – Score every prediction (EM, EX, component F1)

This cell scores each model's SQL for each example with the official Spider code. In plain terms:

- **Exact Match (EM)** checks whether the predicted query is *built the same way* as the gold one. Both queries are split into parts (selected columns, tables, conditions, grouping, ordering, keywords). EM = 1 only if **every** part matches. The order inside a part doesn't matter (`SELECT name, age` = `SELECT age, name`), and **values are ignored** (`WHERE age > 30` matches `WHERE age > 40`). Example: `SELECT count(*) FROM singer` vs gold `SELECT count(*) FROM singer` gives 1, but `SELECT count(Singer_ID) FROM singer` gives 0, even though it returns the same number. EM is strict: it rewards writing the query like the gold query.
- **Execution Accuracy (EX)** runs both queries on the real database and checks that they **return the same rows** (the order only matters if the gold query has `ORDER BY`). The `count(Singer_ID)` example above gets EX = 1. EX is closer to "did the model answer the question", but it can occasionally count a wrong query as right if it happens to return the same rows on this particular database.
- **Component F1** looks at one part at a time (SELECT, WHERE, GROUP BY, ORDER BY, KEYWORDS) and measures how much of that part is right: precision (how much of what the model wrote is correct) and recall (how much of the gold part the model got). It shows *where* the queries go wrong, e.g. a model may get SELECT right but WHERE wrong.

**A limitation of EM and component F1, and our extra column.** Before it can compare parts, the official code has to *read* the SQL with a small parser written in 2018 for the gold queries' style. It doesn't understand some valid SQL, most importantly `INNER JOIN`, which means exactly the same as `JOIN`. When the parser can't read a prediction, the prediction is scored as empty (EM = 0, component F1 = 0), however correct it is. Gemma 4 E4B writes `INNER JOIN` very often, so its official EM is pulled down. We therefore also report **EM (normalized)**: the same official EM, computed after one find-and-replace, `INNER JOIN` → `JOIN`. That rewrite doesn't change the query's meaning. This column is **our addition**, not part of the Spider paper or the official script. Official EM stays the main EM number. Details are in `docs/evaluation_metrics.md`.

**Invalid outputs count as wrong.** If no SQL could be extracted from an output, or the parser can't read it, or it fails when executed, it gets 0 on every metric. The counts of each case are reported in Step 3.4.

The per-example scores are saved to `outputs/metrics/task1_per_example.csv` (one row per model × example). The official execution check uses `asyncio.run`, which can't run inside Jupyter's own event loop, so the scoring runs in worker threads (this only changes where it runs, not how it scores). Scoring takes a few minutes, mostly Gemma 3 1B queries that are slow to execute.

In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor

from evaluation import build_foreign_key_map_from_json, build_valid_col_units, rebuild_sql_col, rebuild_sql_val
from exec_eval import eval_exec_match, exec_on_db, postprocess, remove_distinct

KMAPS = build_foreign_key_map_from_json(str(C.SPIDER_DIR / "tables.json"))
# what evaluate() scores a prediction as when the parser can't read it
EMPTY_SQL = {"except": None, "from": {"conds": [], "table_units": []}, "groupBy": [], "having": [], "intersect": None,
             "limit": None, "orderBy": [], "select": [False, []], "union": None, "where": []}
COMPONENTS = {"select": "SELECT", "where": "WHERE", "group": "GROUP BY", "order": "ORDER BY", "keywords": "KEYWORDS"}
INNER_JOIN = re.compile(r"\bINNER\s+JOIN\b", re.I)


def parse_pred(schema, p_str):
    if not p_str:
        return EMPTY_SQL, False
    try:
        return get_sql(schema, p_str), True
    except Exception:
        return EMPTY_SQL, False


def exact_match(db_id, gold_str, p_sql):
    """Official EM + component scores (same steps as evaluate() in evaluation.py)."""
    schema, kmap = eval_schemas[db_id], KMAPS[db_id]
    g_sql = get_sql(schema, gold_str)
    g_sql = rebuild_sql_col(build_valid_col_units(g_sql["from"]["table_units"], schema), rebuild_sql_val(g_sql), kmap)
    p_sql = rebuild_sql_col(build_valid_col_units(p_sql["from"]["table_units"], schema), rebuild_sql_val(p_sql), kmap)
    ev = Evaluator()
    return int(ev.eval_exact_match(p_sql, g_sql)), ev.partial_scores


def score_example(ex, pred):
    db_id, schema, db = ex["db_id"], eval_schemas[ex["db_id"]], str(db_path(ex["db_id"]))
    p_str = pred.replace("value", "1")  # same preprocessing as evaluate()
    p_sql, parsed = parse_pred(schema, p_str)
    em, partial = exact_match(db_id, ex["query"], p_sql)
    p_sql_norm, parsed_norm = parse_pred(schema, INNER_JOIN.sub("JOIN", p_str))
    em_norm, _ = exact_match(db_id, ex["query"], p_sql_norm)
    if p_str:
        ex_score = eval_exec_match(db=db, p_str=p_str, g_str=ex["query"], plug_value=False, keep_distinct=False,
                                   progress_bar_for_each_datapoint=False)
        exec_error = asyncio.run(exec_on_db(db, remove_distinct(postprocess(p_str))))[0] == "exception"
    else:  # nothing to run: wrong (an empty query would return no rows and could "match" an empty gold result)
        ex_score, exec_error = 0, False
    row = {"em": em, "em_normalized": em_norm, "ex": int(ex_score), "parsed": parsed, "parsed_normalized": parsed_norm,
           "exec_error": exec_error, "uses_inner_join": bool(INNER_JOIN.search(p_str))}
    for c in COMPONENTS:
        s = partial[c]
        row |= {f"{c}_acc": s["acc"], f"{c}_rec": s["rec"], f"{c}_has_pred": s["pred_total"] > 0, f"{c}_has_gold": s["label_total"] > 0}
    return row


C.METRICS_DIR.mkdir(parents=True, exist_ok=True)
rows = []
ids = sorted(inputs)
for key, spec in C.MODELS.items():
    recs = runs[key]
    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=8) as pool:
        scored = list(pool.map(lambda i: score_example(inputs[i], recs[i]["query"]), ids))
    for i, s in zip(ids, scored):
        r = recs[i]
        rows.append({"model": spec.name, "id": i, "difficulty": inputs[i]["difficulty"], "valid": r["valid"],
                     "truncated": r["truncated"], **s, "input_tokens": r["input_tokens"], "output_tokens": r["output_tokens"],
                     "thinking_tokens": r["thinking_tokens"], "latency_s": r["latency_s"]})
    print(f"{spec.name}: scored {len(ids)} examples in {time.perf_counter() - t0:.0f} s")

per_example = pd.DataFrame(rows)
per_example.to_csv(C.METRICS_DIR / "task1_per_example.csv", index=False)
print("saved", C.METRICS_DIR / "task1_per_example.csv", per_example.shape)

## Step 3.3 – Cross-check against the official evaluation script

Step 3.2 calls the official scoring functions one example at a time, so we can keep per-example results. To make sure nothing was lost in that rewiring, this cell also runs the official script the standard way (`evaluation.py --etype all`) on files with the gold and predicted queries. It then checks that its printed EX, EM and component F1 equal ours for every model and difficulty level. An empty prediction is written as the placeholder `SELECT`, which fails the same way an empty query does and keeps the file lines aligned.

In [ ]:
import subprocess
import tempfile

ROW = re.compile(r"^(count|execution|exact match|select|where|group|order|keywords)\s+(.*)$")


def official_scores(model_key):
    recs = runs[model_key]
    with tempfile.TemporaryDirectory() as tmp:
        gold, pred = Path(tmp) / "gold.txt", Path(tmp) / "pred.txt"
        gold.write_text("".join(f"{inputs[i]['query'].strip()}\t{inputs[i]['db_id']}\n" for i in ids))
        pred.write_text("".join((recs[i]["query"] or "SELECT") + "\n" for i in ids))
        out = subprocess.run(
            [sys.executable, "evaluation.py", "--gold", str(gold), "--pred", str(pred), "--etype", "all",
             "--db", str(C.SPIDER_DIR / "database"), "--table", str(C.SPIDER_DIR / "tables.json")],
            cwd=C.EVAL_SCRIPT_DIR, capture_output=True, text=True, check=True).stdout
    found, section = {}, None
    for line in out.splitlines():
        if "PARTIAL MATCHING" in line:
            section = line
        m = ROW.match(line)
        if m and (m.group(1) in ("count", "execution", "exact match") or "F1" in (section or "")):
            found[m.group(1)] = [float(v) for v in m.group(2).split()]
    return found  # metric -> [easy, medium, hard, extra, all]


def component_scores(df, c):
    """Official aggregation: precision over examples with a predicted part, recall over examples with a gold part."""
    acc = df.loc[df[f"{c}_has_pred"], f"{c}_acc"].mean() if df[f"{c}_has_pred"].any() else 0.0
    rec = df.loc[df[f"{c}_has_gold"], f"{c}_rec"].mean() if df[f"{c}_has_gold"].any() else 0.0
    f1 = 1.0 if acc == 0 and rec == 0 else 2 * acc * rec / (acc + rec)  # the official script uses 1.0 when both are 0
    return acc, rec, f1


def level_slice(df, level):
    return df if level == "all" else df[df["difficulty"] == level]


for key, spec in C.MODELS.items():
    off = official_scores(key)
    df = per_example[per_example["model"] == spec.name]
    for k, level in enumerate(LEVELS):
        d = level_slice(df, level)
        assert off["count"][k] == len(d), (spec.name, level)
        assert abs(off["execution"][k] - d["ex"].mean()) < 6e-4, (spec.name, level, "EX")
        assert abs(off["exact match"][k] - d["em"].mean()) < 6e-4, (spec.name, level, "EM")
        for c in COMPONENTS:
            assert abs(off[c][k] - component_scores(d, c)[2]) < 6e-4, (spec.name, level, c)
    print(f"{spec.name}: EX, EM and component F1 match the official script at every difficulty level")

## Step 3.4 – Summary tables

This cell builds the results tables and saves each as a CSV in `outputs/metrics/`. **`task1_summary.csv` is the central table**: one row per model with performance (EX and EM by difficulty and overall, plus normalized EM) and efficiency (time and output tokens per example). It is the main results table for the report. The other files give the supporting detail:
- **`task1_accuracy_by_difficulty.csv`:** EX, official EM and EM (normalized) per model, for each difficulty level and overall, with the number of examples. This is each model compared with the gold queries.
- **`task1_component_f1.csv`:** precision, recall and F1 per SQL part (SELECT, WHERE, GROUP BY, ORDER BY, KEYWORDS), over all examples.
- **`task1_invalid_outputs.csv`:** how many outputs were wrong for technical reasons, all counted as incorrect:
  - no SQL extracted;
  - SQL the official parser couldn't read, and how many of those still returned the right answer when executed;
  - how many parse once `INNER JOIN` → `JOIN`;
  - SQL that failed when executed;
  - outputs cut off at the 8192-token cap.
- **`task1_efficiency.csv`:**
  - inference time per example: mean and median, amortized over batches of 16 on one A40;
  - total run time;
  - output tokens per example: mean, median, max;
  - thinking tokens: E4B only.
- **`task1_efficiency_by_difficulty.csv`:** the same time and token measures for each difficulty level. It shows, for example, whether E4B thinks longer on harder questions.

In [ ]:
def rate(x):
    return round(float(x), 4)


accuracy = pd.DataFrame([
    {"model": name, "difficulty": level, "n": len(d), "EX": rate(d["ex"].mean()), "EM": rate(d["em"].mean()),
     "EM_normalized": rate(d["em_normalized"].mean())}
    for name in MODEL_NAMES for level in LEVELS
    for d in [level_slice(per_example[per_example["model"] == name], level)]
])
components = pd.DataFrame([
    {"model": name, "component": label, **dict(zip(["precision", "recall", "F1"], map(rate, component_scores(d, c))))}
    for name in MODEL_NAMES for c, label in COMPONENTS.items()
    for d in [per_example[per_example["model"] == name]]
])
invalid = pd.DataFrame([
    {"model": name,
     "no_sql_extracted": int((~d["valid"]).sum()),
     "unparseable_by_official_parser": int((d["valid"] & ~d["parsed"]).sum()),
     "unparseable_but_EX_correct": int((d["valid"] & ~d["parsed"] & (d["ex"] == 1)).sum()),
     "uses_INNER_JOIN": int(d["uses_inner_join"].sum()),
     "parseable_after_INNER_JOIN_rewrite": int((d["valid"] & ~d["parsed"] & d["parsed_normalized"]).sum()),
     "execution_error": int(d["exec_error"].sum()),
     "truncated": int(d["truncated"].sum())}
    for name in MODEL_NAMES for d in [per_example[per_example["model"] == name]]
])
efficiency = pd.DataFrame([
    {"model": name,
     "s_per_example_mean": round(d["latency_s"].mean(), 3), "s_per_example_median": round(d["latency_s"].median(), 3),
     "total_time_min": round(d["latency_s"].sum() / 60, 1),
     "output_tokens_mean": round(d["output_tokens"].mean(), 1), "output_tokens_median": float(d["output_tokens"].median()),
     "output_tokens_max": int(d["output_tokens"].max()),
     "thinking_tokens_mean": round(d["thinking_tokens"].mean(), 1) if C.MODELS[key].reasoning else None}
    for key, name in zip(C.MODELS, MODEL_NAMES) for d in [per_example[per_example["model"] == name]]
])

efficiency_by_difficulty = pd.DataFrame([
    {"model": name, "difficulty": level, "n": len(d),
     "s_per_example_mean": round(d["latency_s"].mean(), 3), "s_per_example_median": round(d["latency_s"].median(), 3),
     "output_tokens_mean": round(d["output_tokens"].mean(), 1), "output_tokens_median": float(d["output_tokens"].median()),
     "thinking_tokens_mean": round(d["thinking_tokens"].mean(), 1) if C.MODELS[key].reasoning else None}
    for key, name in zip(C.MODELS, MODEL_NAMES) for level in LEVELS
    for d in [level_slice(per_example[per_example["model"] == name], level)]
])

# central summary table: one row per model, performance + efficiency (the report's main results table)
summary = pd.DataFrame([
    {"model": name,
     **{f"{metric}_{level}": accuracy[(accuracy["model"] == name) & (accuracy["difficulty"] == level)][metric].item()
        for metric in ("EX", "EM") for level in LEVELS},
     "EM_normalized_all": accuracy[(accuracy["model"] == name) & (accuracy["difficulty"] == "all")]["EM_normalized"].item(),
     **efficiency.loc[efficiency["model"] == name, ["s_per_example_mean", "output_tokens_mean", "thinking_tokens_mean"]].iloc[0].to_dict()}
    for name in MODEL_NAMES
])

for fname, table in [("task1_summary", summary), ("task1_accuracy_by_difficulty", accuracy), ("task1_component_f1", components),
                     ("task1_invalid_outputs", invalid), ("task1_efficiency", efficiency),
                     ("task1_efficiency_by_difficulty", efficiency_by_difficulty)]:
    table.to_csv(C.METRICS_DIR / f"{fname}.csv", index=False)

print("Summary (central results table, task1_summary.csv):")
display(summary.set_index("model").T)
print("Accuracy by difficulty (EX / EM / EM normalized):")
display(accuracy.pivot(index="model", columns="difficulty", values=["EX", "EM", "EM_normalized"]).reindex(MODEL_NAMES)
        .reindex(columns=LEVELS, level=1))
print("Component F1 (all examples):")
display(components.pivot(index="model", columns="component", values="F1").reindex(MODEL_NAMES)[list(COMPONENTS.values())])
print("Invalid / unparseable outputs (all counted as incorrect):")
display(invalid.set_index("model"))
print("Efficiency:")
display(efficiency.set_index("model"))
print("Efficiency by difficulty (mean s/example, mean output tokens, mean thinking tokens):")
display(efficiency_by_difficulty.pivot(index="model", columns="difficulty",
                                       values=["s_per_example_mean", "output_tokens_mean", "thinking_tokens_mean"])
        .reindex(MODEL_NAMES).reindex(columns=LEVELS, level=1))

## Step 3.5 – Comparing the models with each other

All three models answered the **same** 1034 questions, so we can compare them question by question, not only by their overall scores. For each pair of models (A vs B) and each metric (EX and EM), at each difficulty level and overall, this cell counts:
- **both right:** questions both models got right;
- **only A right / only B right:** questions where they disagree, which is what drives the accuracy gap;
- **both wrong:** questions neither model got right;
- **difference:** A's accuracy minus B's accuracy.

This comparison is descriptive (no significance tests). It is saved to `outputs/metrics/task1_model_agreement.csv`.

In [ ]:
from itertools import combinations

wide = {m: per_example.pivot(index="id", columns="model", values=m) for m in ("ex", "em")}
agreement = []
for a, b in combinations(MODEL_NAMES, 2):
    for metric, table in wide.items():
        for level in LEVELS:
            ids_lvl = [i for i in ids if level == "all" or inputs[i]["difficulty"] == level]
            ra, rb = table.loc[ids_lvl, a].astype(bool), table.loc[ids_lvl, b].astype(bool)
            agreement.append({"model_A": a, "model_B": b, "metric": metric.upper(), "difficulty": level, "n": len(ids_lvl),
                              "acc_A": rate(ra.mean()), "acc_B": rate(rb.mean()), "difference_A_minus_B": rate(ra.mean() - rb.mean()),
                              "both_right": int((ra & rb).sum()), "only_A_right": int((ra & ~rb).sum()),
                              "only_B_right": int((~ra & rb).sum()), "both_wrong": int((~ra & ~rb).sum())})
agreement = pd.DataFrame(agreement)
agreement.to_csv(C.METRICS_DIR / "task1_model_agreement.csv", index=False)
display(agreement[agreement["difficulty"] == "all"].set_index(["model_A", "model_B", "metric"]))

## Step 3.6 – Plots for the report

This cell draws three figures and saves them as PNG files in `outputs/metrics/figures/`. Each model keeps the same color in every figure (1B blue, 4B orange, E4B green-aqua; a colorblind-safe palette). Every bar is labelled with its value, so no number depends on color alone, and the exact numbers are also in the CSV tables.
1. **`task1_accuracy_by_difficulty.png`:** EX, official EM and EM (normalized) by difficulty, one panel per metric on the same 0–1 scale.
2. **`task1_accuracy_vs_time.png`:** overall EX against inference time per example (log scale), showing the accuracy vs. speed trade-off.
3. **`task1_invalid_outputs.png`:** for each model, how many predictions the official parser couldn't read, split into those that were still correct when executed and those that were not. It shows how much the parser limitation affects official EM.

In [ ]:
import matplotlib.pyplot as plt

COLORS = dict(zip(MODEL_NAMES, ["#2a78d6", "#eb6834", "#1baf7a"]))  # fixed categorical order (validated palette)
INK, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#e1e0d9", "#c3c2b7"
plt.rcParams.update({"font.size": 10, "axes.edgecolor": AXIS, "axes.labelcolor": MUTED, "xtick.color": MUTED,
                     "ytick.color": MUTED, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})
C.FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def style_axis(ax):
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)


# 1. accuracy by difficulty: one panel per metric (small multiples, shared 0–1 scale)
panels = [("EX", "Execution accuracy (EX)"), ("EM", "Exact match (EM), official"),
          ("EM_normalized", "EM, normalized (INNER JOIN → JOIN)")]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), sharey=True)
width = 0.26
for ax, (col, title) in zip(axes, panels):
    for k, name in enumerate(MODEL_NAMES):
        vals = [accuracy[(accuracy["model"] == name) & (accuracy["difficulty"] == lvl)][col].item() for lvl in LEVELS]
        xs = [x + (k - 1) * width for x in range(len(LEVELS))]
        bars = ax.bar(xs, vals, width=width, color=COLORS[name], edgecolor="white", linewidth=1.5, label=name)
        ax.bar_label(bars, labels=[f"{v:.2f}" for v in vals], fontsize=7, color=MUTED, padding=2)
    ax.set_xticks(range(len(LEVELS)), [l if l != "all" else "overall" for l in LEVELS])
    ax.set_title(title, color=INK, fontsize=11, loc="left")
    ax.set_ylim(0, 1.05)
    style_axis(ax)
axes[0].set_ylabel("share of examples correct")
axes[0].legend(frameon=False, loc="upper right", fontsize=9)
fig.tight_layout()
fig.savefig(C.FIGURES_DIR / "task1_accuracy_by_difficulty.png", dpi=200, bbox_inches="tight")
plt.show()

# 2. accuracy vs inference time
fig, ax = plt.subplots(figsize=(6.5, 4.2))
for name in MODEL_NAMES:
    x = efficiency.loc[efficiency["model"] == name, "s_per_example_mean"].item()
    y = accuracy[(accuracy["model"] == name) & (accuracy["difficulty"] == "all")]["EX"].item()
    ax.scatter(x, y, s=90, color=COLORS[name], edgecolor="white", linewidth=2, zorder=3)
    ax.annotate(f"{name}\nEX {y:.3f}, {x:.2f} s", (x, y), textcoords="offset points", xytext=(10, -4), fontsize=9, color=INK)
ax.set_xscale("log")
ax.set_xlim(0.08, 10)
ax.set_xticks([0.1, 0.3, 1, 3, 10], ["0.1", "0.3", "1", "3", "10"])
ax.minorticks_off()
ax.set_ylim(0, 1)
ax.set_xlabel("inference time per example (s, log scale; batch 16, one A40)")
ax.set_ylabel("execution accuracy (overall)")
ax.set_title("Accuracy vs. inference time", color=INK, fontsize=11, loc="left")
style_axis(ax)
fig.tight_layout()
fig.savefig(C.FIGURES_DIR / "task1_accuracy_vs_time.png", dpi=200, bbox_inches="tight")
plt.show()

# 3. unparseable predictions: still correct when executed vs not
fig, ax = plt.subplots(figsize=(7, 3.6))
for k, name in enumerate(MODEL_NAMES):
    row = invalid[invalid["model"] == name].iloc[0]
    ok, total = row["unparseable_but_EX_correct"], row["unparseable_by_official_parser"]
    ax.barh(k, ok, color=COLORS[name], edgecolor="white", linewidth=1.5)
    ax.barh(k, total - ok, left=ok, color="white", edgecolor=COLORS[name], linewidth=1.5, hatch="///")
    ax.text(total + 8, k, f"{total} unparseable ({ok} still EX-correct)", va="center", fontsize=9, color=INK)
ax.set_yticks(range(len(MODEL_NAMES)), MODEL_NAMES)
ax.invert_yaxis()
ax.set_xlim(0, 700)
ax.set_xlabel("predictions the official Spider parser could not read (scored EM = 0)")
ax.set_title("Unparseable predictions: solid = still correct when executed, hatched = not", color=INK, fontsize=10, loc="left")
ax.grid(axis="x", color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(C.FIGURES_DIR / "task1_invalid_outputs.png", dpi=200, bbox_inches="tight")
plt.show()
print("saved figures to", C.FIGURES_DIR)